## Web検索連動チャットボットを作成しよう

In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # 1) LLM を用意（ツール呼び出しを有効化）
    llm = ChatOpenAI(model=model_name, temperature=0)

    # 2) Web検索ツール（Tavily）
    search_tool = TavilySearchResults(max_results=5)
    tools = [search_tool]

    # 3) ツールノード（LLMがtool_callsしたらここで実行される）
    tool_node = ToolNode(tools)

    # 4) アシスタントノード（LLMにメッセージを渡して応答を作る）
    #    bind_tools しておくと、必要に応じて tool_calls を返すようになります。
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        # state["messages"] は add_messages により蓄積される
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    # 5) StateGraph を組み立て
    graph_builder = StateGraph(State)
    graph_builder.add_node("assistant", assistant)
    graph_builder.add_node("tools", tool_node)

    # 開始点 → assistant
    graph_builder.set_entry_point("assistant")

    # assistant の出力に tool_calls があれば tools へ、なければ終了へ
    graph_builder.add_conditional_edges(
        "assistant",
        tools_condition,
        # tools_condition が返すキーに対応する遷移先
        {
            "tools": "tools",
            "__end__": "__end__",
        },
    )

    # tools 実行後は assistant に戻して、検索結果を踏まえて最終回答させる
    graph_builder.add_edge("tools", "assistant")

    # 6) 会話スレッド（thread_id）ごとに状態を保持するためのメモリ
    checkpointer = MemorySaver()

    # 7) コンパイル
    graph = graph_builder.compile(checkpointer=checkpointer)
    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        last = event["messages"][-1]
        # tool 実行結果(JSON)を表示しない
        if getattr(last, "type", None) == "ai":
            print(last.content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんばんは！今日はどんなことをお話ししましょうか？
1たす2は3です。何か他にお手伝いできることはありますか？
5かける7は35です。他に計算や質問があれば教えてください！

台湾観光に関する情報をいくつかご紹介します。

1. **台湾観光おすすめ42選**  
   [KKday](https://www.kkday.com/ja/blog/48762/taiwan-sightseeing?srsltid=AfmBOoqEsoWVifYRjCvE-_-GG69uhbrLnySOy879B6S6ZUDObUkZiM8Z)では、台北の観光スポットやモデルコースを紹介しています。台北101、龍山寺、士林夜市、国立故宮博物院など、人気の観光地がリストアップされています。

2. **3泊4日のおすすめモデルコース**  
   [KKday](https://www.kkday.com/ja/blog/12042/34-asia-taiwan-taipei-4day-travelplan?srsltid=AfmBOopmyX4E3qraahDO1771nu_AyUUcaHzUeZXhsDaXrcIk2ZMLIG7J)では、台湾旅行のモデルコースを提案しています。台北市内観光や夜市でのグルメ体験、九份や十分の観光など、充実したプランが組まれています。

3. **台北観光のおすすめスポット19選**  
   [近畿日本ツーリスト](https://www.knt.co.jp/travelguide/kaigai/027/)では、台北の観光スポットを紹介しています。台北101や国立故宮博物院、士林観光夜市など、観光名所が詳しく説明されています。

4. **台湾の現地ツアー／アクティビティ**  
   [VELTRA](https://www.veltra.com/jp/asia/taiwan/)では、台湾での観光ツアーやアクティビティを予約できます。歴史探訪や美術館巡り、グルメツアーなど多彩な選択肢があります。

5. **台湾観光おすすめ49選**  
   [アソビュー](https://www.asoview.com/note/4674/)では、台湾の観光スポットを49選紹介しています。特に九份や士林夜市、台北101など、人気のスポットが詳しく解説されています。